# CropSense — 02 Tool-call distillation dataset (Gemini teacher → verified traces)

Builds the training data for the small on-device / home-GPU assistant (notebook 03).
Gemini answers farmer questions **using the app's real tools**; every trace
(question → tool call → real tool result → final answer) is checked, and only passing traces are kept.

**Runtime: CPU is enough** (Runtime → Change runtime type → CPU). This notebook uses **~0 compute units**;
the cost is Gemini API calls (roughly 3 calls per trace).

**Before running:**
1. On your machine: `cd python && python scripts/export_distill_snapshot.py --out ../colab/snapshot --with-transcripts`
2. Upload the `snapshot/` folder to Drive at `MyDrive/cropsense/snapshot/`.
3. Colab → 🔑 Secrets → add `GEMINI_API_KEY` (enable notebook access).
4. Run all. It is **resumable**: re-running skips questions already done.

Output in `MyDrive/cropsense/distill_v1/`: `train.jsonl`, `val.jsonl`, `tools.json`, `rejected.jsonl`, `stats.json`.

> Farm/plot tools use **synthetic farm profiles** (no real user data leaves your DB). The student only
> needs to learn *when and how* to call them; real values come from the backend at run time.

In [ ]:
# ---- Config ----
TEACHER_MODEL = 'gemini-3.8-flash'      # same as backend GEMINI_MODEL
HELPER_MODEL  = 'gemini-3.5-flash-lite' # question writing + judge (cheap)
N_QUESTIONS   = 3000                    # start with ~300 to check quality and cost, then raise
LANGS = {'en': 'English', 'hi': 'Hindi', 'mr': 'Marathi', 'pa': 'Punjabi'}
USE_JUDGE  = True
JUDGE_MIN  = 4          # keep traces the judge scores >= this (1-5)
WORKERS    = 6          # parallel teacher calls; lower if you hit 429s
DRIVE_ROOT   = '/content/drive/MyDrive/cropsense'
SNAPSHOT_DIR = f'{DRIVE_ROOT}/snapshot'
OUT_DIR      = f'{DRIVE_ROOT}/distill_v1'
SEED = 7

In [ ]:
!pip -q install "google-genai>=1.30" jsonschema
import json, os, random, re, time, math, threading, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive, userdata
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
from google import genai
from google.genai import types
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
random.seed(SEED)

## 1. Load the snapshot

In [ ]:
def load(name):
    p = Path(SNAPSHOT_DIR) / f'{name}.json'
    if not p.exists():
        print(f'  (missing {name}.json, its tool will return "no data")'); return []
    rows = json.loads(p.read_text()); print(f'  {name}: {len(rows)} rows'); return rows

PRICES, MSP, PESTS, VETS, TRANSCRIPTS = (load(n) for n in
    ['market_prices', 'msp_rates', 'pest_disease_data', 'veterinarians', 'transcripts'])

def _s(v): return '' if v is None else (json.dumps(v, ensure_ascii=False) if isinstance(v, (list, dict)) else str(v))

## 2. Tools
Names match `python/app/agents/agent_tools.py` where a tool already exists
(`get_farm_details`, `get_soil_info`, `get_weather_forecast`, `get_market_prices`).
`get_msp_rate`, `search_pest_disease` and `find_veterinarian` are new and must be added to the backend
before the student model is used there. Weather uses the free Open-Meteo API (real data, no key).

In [ ]:
STATES = {  # synthetic farm locations: (state, district, lat, lon)
    'Maharashtra': [('Pune', 18.52, 73.86), ('Nashik', 19.99, 73.79), ('Nagpur', 21.15, 79.09)],
    'Punjab': [('Ludhiana', 30.90, 75.85), ('Bathinda', 30.21, 74.95)],
    'Uttar Pradesh': [('Lucknow', 26.85, 80.95), ('Meerut', 28.98, 77.71)],
    'Madhya Pradesh': [('Indore', 22.72, 75.86), ('Bhopal', 23.26, 77.41)],
    'Rajasthan': [('Jaipur', 26.91, 75.79)], 'Gujarat': [('Anand', 22.56, 72.95)],
    'Bihar': [('Patna', 25.59, 85.14)], 'Haryana': [('Karnal', 29.69, 76.99)],
}
SOILS = ['Black (Regur)', 'Alluvial', 'Red', 'Laterite', 'Sandy loam']
FARMS, PLOTS = {}, {}
for fid in range(101, 161):
    st = random.choice(list(STATES)); d, lat, lon = random.choice(STATES[st])
    FARMS[fid] = dict(farm_id=fid, state=st, district=d, latitude=lat, longitude=lon,
                      total_area_acres=round(random.uniform(0.5, 12), 1), soil_type=random.choice(SOILS),
                      irrigation=random.choice(['Canal', 'Borewell', 'Drip', 'Rainfed']))
    for k in range(random.randint(1, 3)):
        pid = fid * 10 + k
        PLOTS[pid] = dict(plot_id=pid, farm_id=fid, area_acres=round(random.uniform(0.3, 4), 1),
            soil_test=None if random.random() < 0.2 else dict(
                nitrogen_kg_per_ha=random.randint(120, 480), phosphorus_kg_per_ha=random.randint(8, 40),
                potassium_kg_per_ha=random.randint(100, 420), ph=round(random.uniform(5.6, 8.6), 1),
                organic_carbon_percent=round(random.uniform(0.2, 1.1), 2), ec_ds_per_m=round(random.uniform(0.1, 1.8), 2)))

def norm(s): return re.sub(r'\s+', ' ', _s(s).lower()).strip()

def get_farm_details(farm_id: int):
    f = FARMS.get(int(farm_id)); return f or {'error': f'Farm {farm_id} not found'}

def get_soil_info(plot_id: int):
    p = PLOTS.get(int(plot_id))
    if not p: return {'error': f'Plot {plot_id} not found'}
    return p if p['soil_test'] else {**p, 'soil_test': 'No soil test records for this plot'}

_wx_cache = {}
def get_weather_forecast(latitude: float, longitude: float):
    key = (round(float(latitude), 2), round(float(longitude), 2))
    if key not in _wx_cache:
        url = ('https://api.open-meteo.com/v1/forecast?latitude=%s&longitude=%s&timezone=Asia%%2FKolkata'
               '&forecast_days=5&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,relative_humidity_2m_mean' % key)
        try:
            d = json.load(urllib.request.urlopen(url, timeout=20))['daily']
            _wx_cache[key] = [dict(date=d['time'][i], min_temp_c=d['temperature_2m_min'][i], max_temp_c=d['temperature_2m_max'][i],
                                   rain_mm=d['precipitation_sum'][i], humidity_pct=d.get('relative_humidity_2m_mean', [None]*5)[i])
                              for i in range(len(d['time']))]
        except Exception as e:
            return {'error': f'weather service unavailable: {e}'}
    return {'forecast': _wx_cache[key]}

def get_market_prices(crop_name: str, state: str = None, district: str = None):
    q = norm(crop_name)
    rows = [r for r in PRICES if q and q in norm(r.get('item_name'))]
    if state:    rows = [r for r in rows if norm(state) in norm(r.get('state'))] or rows
    if district: rows = [r for r in rows if norm(district) in norm(r.get('district'))] or rows
    rows = sorted(rows, key=lambda r: _s(r.get('transaction_date')), reverse=True)[:5]
    return {'prices': rows} if rows else {'error': f"No price listings found for '{crop_name}'"}

def get_msp_rate(crop_name: str, year: int = None):
    q = norm(crop_name)
    rows = [r for r in MSP if q and (q in norm(r.get('crop_name')) or norm(r.get('crop_name')) in q)]
    if year: rows = [r for r in rows if int(r.get('year') or 0) == int(year)] or rows
    rows = sorted(rows, key=lambda r: int(r.get('year') or 0), reverse=True)[:3]
    return {'msp': rows} if rows else {'error': f"No MSP found for '{crop_name}' (MSP covers notified crops only)"}

def search_pest_disease(query: str, crop: str = None):
    words = [w for w in norm(query).split() if len(w) > 2]
    def score(r):
        text = norm(' '.join(_s(r.get(k)) for k in ('name', 'scientific_name', 'description', 'affected_crops', 'type')))
        s = sum(w in text for w in words)
        if crop and norm(crop) in norm(r.get('affected_crops')): s += 2
        return s
    hits = sorted(PESTS, key=score, reverse=True)
    hits = [h for h in hits[:3] if score(h) > 0]
    return {'matches': hits} if hits else {'error': 'No matching pest or disease in the knowledge base'}

def find_veterinarian(state: str, district: str = None, species: str = None):
    rows = [r for r in VETS if norm(state) in norm(r.get('location_state'))]
    if district: rows = [r for r in rows if norm(district) in norm(r.get('location_district'))] or rows
    if species:  rows = [r for r in rows if norm(species) in norm(r.get('species_supported'))] or rows
    rows = sorted(rows, key=lambda r: (not r.get('verified'), -(r.get('rating') or 0)))[:3]
    return {'vets': rows, 'note': 'Contact details are shown in the app'} if rows else {'error': f'No vets listed for {state}'}

S, I, N = {'type': 'string'}, {'type': 'integer'}, {'type': 'number'}
TOOL_SPECS = [
 ('get_farm_details', 'Get the farmer\'s farm: state, district, coordinates, area, soil type, irrigation.', {'farm_id': I}, ['farm_id']),
 ('get_soil_info', 'Get the latest soil test (N, P, K kg/ha, pH, organic carbon, EC) for a farm plot.', {'plot_id': I}, ['plot_id']),
 ('get_weather_forecast', '5-day weather forecast (temperature, rain, humidity) for coordinates.', {'latitude': N, 'longitude': N}, ['latitude', 'longitude']),
 ('get_market_prices', 'Latest mandi/market prices for a crop or produce, optionally filtered by state/district.', {'crop_name': S, 'state': S, 'district': S}, ['crop_name']),
 ('get_msp_rate', 'Government Minimum Support Price (MSP) for a crop, per quintal and per kg.', {'crop_name': S, 'year': I}, ['crop_name']),
 ('search_pest_disease', 'Search the pest & disease knowledge base by symptom or name; returns treatments and prevention.', {'query': S, 'crop': S}, ['query']),
 ('find_veterinarian', 'Find veterinarians by state, district and animal species.', {'state': S, 'district': S, 'species': S}, ['state']),
]
TOOLS_JSON = [{'type': 'function', 'function': {'name': n, 'description': d,
    'parameters': {'type': 'object', 'properties': p, 'required': r}}} for n, d, p, r in TOOL_SPECS]
TOOL_FNS = {n: globals()[n] for n, *_ in TOOL_SPECS}
GEMINI_TOOLS = [types.Tool(function_declarations=[types.FunctionDeclaration(
    name=t['function']['name'], description=t['function']['description'],
    parameters_json_schema=t['function']['parameters']) for t in TOOLS_JSON])]
Path(OUT_DIR, 'tools.json').write_text(json.dumps(TOOLS_JSON, ensure_ascii=False, indent=1))
print(get_msp_rate('wheat')); print(search_pest_disease('yellow leaves wheat'))

## 3. System prompt (shared by teacher and student)

In [ ]:
SYSTEM = """You are CropSense Sahayak, a farming assistant for Indian farmers (crops and livestock).
Rules:
- Reply in the farmer's language (English, Hindi, Marathi or Punjabi), in short simple sentences.
- Facts like prices, MSP, doses, soil values, weather and vet names MUST come from a tool result. Never invent numbers.
- If a tool returns an error or no data, say so plainly and suggest a next step (e.g. ask the local KVK or agriculture officer).
- If you need information you don't have (crop, district, which animal), ask ONE short question.
- Chemical treatments: give the product type and say to follow the label dose; mention safety (gloves, mask, waiting period).
- Keep answers under 120 words unless the farmer asks for detail."""

def system_for(ctx):
    s = SYSTEM
    if ctx.get('farm_id'): s += f"\nThe signed-in farmer's farm_id is {ctx['farm_id']}" + (f"; plot_ids: {ctx['plot_ids']}" if ctx.get('plot_ids') else '') + '.'
    return s

## 4. Write farmer questions (seeded from your real data)

In [ ]:
crop_names = sorted({r.get('crop_name') for r in MSP if r.get('crop_name')} | {r.get('item_name') for r in PRICES[:5000] if r.get('item_name')}) or ['Wheat', 'Paddy', 'Cotton', 'Soybean', 'Onion', 'Tur']
pest_names = [r.get('name') for r in PESTS if r.get('name')] or ['Fall armyworm', 'Yellow rust', 'Pink bollworm']
vet_places = sorted({(r.get('location_state'), r.get('location_district')) for r in VETS if r.get('location_state')}) or [('Maharashtra', 'Pune')]

TOPICS = [  # (topic, weight, hint builder, needs farm context)
 ('market_price', 3, lambda: f"price of {random.choice(crop_names)} in {random.choice(list(STATES))}", False),
 ('msp',          2, lambda: f"MSP of {random.choice(crop_names)}", False),
 ('pest',         3, lambda: f"symptoms that match {random.choice(pest_names)} (describe symptoms, don't always name it)", False),
 ('vet',          2, lambda: "a sick %s near %s, %s" % ((random.choice(['cow', 'buffalo', 'goat', 'poultry', 'sheep']),) + tuple(random.choice(vet_places))), False),
 ('weather',      2, lambda: "whether to spray / irrigate / harvest this week given the weather at my farm", True),
 ('soil',         2, lambda: "fertilizer or soil health of my plot", True),
 ('multi',        2, lambda: f"should I sell {random.choice(crop_names)} now or hold (compare mandi price with MSP)", False),
 ('clarify',      1, lambda: "a vague question missing the crop or the place (assistant should ask one question)", False),
 ('general',      1, lambda: "a greeting or general farming question that needs no data", False),
]
if TRANSCRIPTS: TOPICS.append(('real', 3, None, False))

def write_questions(topic, hint, lang, n=10):
    prompt = (f"Write {n} different, realistic questions an Indian farmer would type or say to a farming app, in {LANGS[lang]}"
              f"{' (native script)' if lang != 'en' else ''}. Topic: {hint}. Vary tone, length, spelling mistakes and detail. "
              "Some short like voice notes. Return only a JSON list of strings.")
    r = client.models.generate_content(model=HELPER_MODEL, contents=prompt,
          config=types.GenerateContentConfig(temperature=1.0, response_mime_type='application/json'))
    return [q for q in json.loads(r.text) if isinstance(q, str) and q.strip()][:n]

qpath = Path(OUT_DIR, 'questions.jsonl')
questions = [json.loads(l) for l in qpath.open()] if qpath.exists() else []
weights = [w for _, w, _, _ in TOPICS]
while len(questions) < N_QUESTIONS:
    topic, _, hint, needs_farm = random.choices(TOPICS, weights)[0]
    lang = random.choice(list(LANGS))
    try:
        if topic == 'real':
            batch = [t['transcript'] for t in random.sample(TRANSCRIPTS, min(10, len(TRANSCRIPTS)))]
        else:
            batch = write_questions(topic, hint(), lang)
    except Exception as e:
        print('retry:', e); time.sleep(5); continue
    with qpath.open('a') as f:
        for q in batch:
            ctx = {}
            if needs_farm or random.random() < 0.3:
                fid = random.choice(list(FARMS)); ctx = {'farm_id': fid, 'plot_ids': [p for p in PLOTS if PLOTS[p]['farm_id'] == fid]}
            rec = {'id': f'q{len(questions):06d}', 'topic': topic, 'lang': lang, 'question': q, 'ctx': ctx}
            questions.append(rec); f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    print(f'\r{len(questions)}/{N_QUESTIONS} questions', end='')
print(); print(random.sample(questions, 3))

## 5. Teacher: Gemini answers with real tool calls (resumable)

In [ ]:
def call_with_retry(**kw):
    for attempt in range(6):
        try: return client.models.generate_content(**kw)
        except Exception as e:
            if attempt == 5: raise
            time.sleep(min(60, 4 * 2 ** attempt) if '429' in str(e) or '503' in str(e) else 3)

def run_teacher(rec, max_rounds=4):
    cfg = types.GenerateContentConfig(system_instruction=system_for(rec['ctx']), tools=GEMINI_TOOLS, temperature=0.4,
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
    contents = [types.Content(role='user', parts=[types.Part.from_text(text=rec['question'])])]
    steps = []  # [{'calls':[{name,arguments}], 'results':[...]}]
    for _ in range(max_rounds):
        r = call_with_retry(model=TEACHER_MODEL, contents=contents, config=cfg)
        content = r.candidates[0].content
        contents.append(content)  # keeps thought signatures for the next round
        calls = [p.function_call for p in (content.parts or []) if p.function_call]
        if not calls:
            text = ''.join(p.text for p in (content.parts or []) if p.text and not getattr(p, 'thought', False)).strip()
            return {**rec, 'steps': steps, 'final': text}
        step = {'calls': [], 'results': []}; parts = []
        for c in calls:
            args = dict(c.args or {})
            try: out = TOOL_FNS[c.name](**args) if c.name in TOOL_FNS else {'error': f'unknown tool {c.name}'}
            except Exception as e: out = {'error': f'bad arguments: {e}'}
            step['calls'].append({'name': c.name, 'arguments': args}); step['results'].append(out)
            parts.append(types.Part.from_function_response(name=c.name, response={'result': out}))
        steps.append(step); contents.append(types.Content(role='user', parts=parts))
    return {**rec, 'steps': steps, 'final': '', 'error': 'too many tool rounds'}

tpath = Path(OUT_DIR, 'traces.jsonl')
done = {json.loads(l)['id'] for l in tpath.open()} if tpath.exists() else set()
todo = [q for q in questions if q['id'] not in done]
print(f'{len(done)} done, {len(todo)} to go'); lock = threading.Lock()
with ThreadPoolExecutor(WORKERS) as ex, tpath.open('a') as f:
    futs = {ex.submit(run_teacher, q): q for q in todo}
    for i, fu in enumerate(as_completed(futs), 1):
        try: tr = fu.result()
        except Exception as e: tr = {**futs[fu], 'steps': [], 'final': '', 'error': str(e)[:300]}
        with lock: f.write(json.dumps(tr, ensure_ascii=False, default=str) + '\n'); f.flush()
        if i % 25 == 0: print(f'\r{i}/{len(todo)}', end='')
print('\nteacher done')

## 6. Verify
Rule checks first (cheap), then an optional AI judge:
1. Every tool call uses a known tool and its arguments match the schema.
2. Topics that need data actually called a tool; `general`/`clarify` didn't need one.
3. **No invented numbers:** every number ≥ 10 in the answer must appear in the question or a tool result.
4. Answer is in the right script (Devanagari for Hindi/Marathi, Gurmukhi for Punjabi).
5. Judge (flash-lite) scores faithfulness to the tool results, 1–5.

In [ ]:
import jsonschema
SCHEMAS = {t['function']['name']: t['function']['parameters'] for t in TOOLS_JSON}
NEEDS_TOOL = {'market_price', 'msp', 'pest', 'vet', 'weather', 'soil', 'multi'}
DIGITS = str.maketrans('०१२३४५६७८९੦੧੨੩੪੫੬੭੮੯', '01234567890123456789')

def numbers(text):
    t = _s(text).translate(DIGITS).replace(',', '')
    return {float(x) for x in re.findall(r'\d+(?:\.\d+)?', t)}

def script_ok(text, lang):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters: return False
    if lang == 'en': return sum(ch.isascii() for ch in letters) / len(letters) > 0.8
    lo, hi = (0x0A00, 0x0A7F) if lang == 'pa' else (0x0900, 0x097F)
    return sum(lo <= ord(ch) <= hi for ch in letters) / len(letters) > 0.5

def rule_check(tr):
    if tr.get('error') or not tr.get('final'): return tr.get('error') or 'empty answer'
    calls = [c for s in tr['steps'] for c in s['calls']]
    for c in calls:
        if c['name'] not in SCHEMAS: return f"unknown tool {c['name']}"
        try: jsonschema.validate(c['arguments'], SCHEMAS[c['name']])
        except jsonschema.ValidationError as e: return f"bad args {c['name']}: {e.message}"
    if tr['topic'] in NEEDS_TOOL and not calls and '?' not in tr['final']: return 'no tool call for a data question'
    allowed = numbers(tr['question']) | numbers(json.dumps([s['results'] for s in tr['steps']], ensure_ascii=False, default=str))
    allowed |= {round(a / 100, 2) for a in allowed} | {round(a * 100, 2) for a in allowed}  # quintal <-> kg
    bad = [n for n in numbers(tr['final']) if n >= 10 and n not in allowed and not 2020 <= n <= 2030]
    if bad: return f'invented numbers {bad[:5]}'
    if tr['topic'] != 'real' and not script_ok(tr['final'], tr['lang']): return 'wrong language/script'
    return None

def judge(tr):
    prompt = ("Score this farming-assistant answer 1-5 for FAITHFULNESS and HELPFULNESS. 5 = every fact comes from the tool "
              "results or is safe general advice, it answers the question, right language, no harmful dose advice. "
              "1 = invented facts or unsafe. Return JSON {\"score\": int, \"reason\": str}.\n\n"
              f"QUESTION: {tr['question']}\nTOOL RESULTS: {json.dumps([s for s in tr['steps']], ensure_ascii=False, default=str)[:6000]}\n"
              f"ANSWER: {tr['final']}")
    r = call_with_retry(model=HELPER_MODEL, contents=prompt,
          config=types.GenerateContentConfig(temperature=0, response_mime_type='application/json'))
    return json.loads(r.text)

traces = [json.loads(l) for l in tpath.open()]
traces = list({t['id']: t for t in traces}.values())
kept, rejected = [], []
for t in traces:
    why = rule_check(t)
    (rejected if why else kept).append({**t, 'reject': why} if why else t)
print(f'rule checks: kept {len(kept)}, rejected {len(rejected)}')
if USE_JUDGE:
    with ThreadPoolExecutor(WORKERS) as ex:
        def safe_judge(t):
            try: return judge(t)
            except Exception as e: return {'score': 0, 'reason': f'judge failed: {e}'}
        scores = list(ex.map(safe_judge, kept))
    k2 = []
    for t, j in zip(kept, scores):
        if int(j.get('score') or 0) >= JUDGE_MIN: k2.append({**t, 'judge': j})
        else: rejected.append({**t, 'reject': 'judge: ' + str(j.get('reason'))[:200]})
    kept = k2
    print(f'after judge: kept {len(kept)}')
from collections import Counter
print('reject reasons:', Counter(re.sub(r'\[.*', '', r['reject'])[:40] for r in rejected).most_common(8))

## 7. Export chat-format training data (OpenAI / Qwen tool-call format)

In [ ]:
def to_messages(t):
    msgs = [{'role': 'system', 'content': system_for(t['ctx'])}, {'role': 'user', 'content': t['question']}]
    for s in t['steps']:
        msgs.append({'role': 'assistant', 'content': '', 'tool_calls': [
            {'type': 'function', 'function': {'name': c['name'], 'arguments': c['arguments']}} for c in s['calls']]})
        for c, r in zip(s['calls'], s['results']):
            msgs.append({'role': 'tool', 'name': c['name'], 'content': json.dumps(r, ensure_ascii=False, default=str)})
    msgs.append({'role': 'assistant', 'content': t['final']})
    return {'id': t['id'], 'topic': t['topic'], 'lang': t['lang'], 'messages': msgs}

random.shuffle(kept)
n_val = max(20, len(kept) // 20)
for name, rows in [('val', kept[:n_val]), ('train', kept[n_val:])]:
    with Path(OUT_DIR, f'{name}.jsonl').open('w') as f:
        for t in rows: f.write(json.dumps(to_messages(t), ensure_ascii=False, default=str) + '\n')
with Path(OUT_DIR, 'rejected.jsonl').open('w') as f:
    for t in rejected: f.write(json.dumps(t, ensure_ascii=False, default=str) + '\n')
stats = {'questions': len(questions), 'traces': len(traces), 'kept': len(kept), 'val': n_val,
         'by_topic': Counter(t['topic'] for t in kept), 'by_lang': Counter(t['lang'] for t in kept)}
Path(OUT_DIR, 'stats.json').write_text(json.dumps(stats, indent=1)); print(json.dumps(stats, indent=1))

**Before training:** open `rejected.jsonl` and read ~20 rejects and ~20 kept examples. If the number check rejects good
answers (e.g. unit conversions), loosen it; if the judge lets bad doses through, raise `JUDGE_MIN`.
Then run **03_finetune_tool_model.ipynb** on an L4.